## 1 - Data Analysis

El objetivo principal es extraer y analizar los datos de la base de datos ***MIMIC-IV***
El primer paso es analizar de que tipo de datos disponemos, de primeras se puede observar que el formato de los archivos es `csv.gz`, el formato `gz` indica que es un archivo comprimido, y previamente tiene el formato `csv`, por lo tanto se puede llegar a suponer que es un archivo csv comprimido. La libreria pandas ayudara a leer los archivos csv comprimidos directamente sin descompression manual.

Lo prinicipal es importar las librerias que seran necesarias para analizar y procesar estos datos

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

Con las librerias requeridas cargadas ya se pueden importar y empezar a usar los datos, aunque antes hay que entender como estan estructurados y que relacion hay entre diferentes archivos `csv`.

Segun la documentacion de **MIMIC-IV** tenemos dos directorios principales:

- **hosp**: Contiene **546,028** registros unicos de hospitalizaciones para **223,452** individiuos unicos. Estos datos se dividen en cuatro categorias principales:
    - **Información Básica y Logística**: Incluye los datos personales de los pacientes (edad, genero, etc), y registra cuando entra y sale un paciente de una unidad en concreto.
    - **Datos Clínicos y de Laboratorio**: Contiene eventos registrados en un laboratorio, como por ejemplo resultados de analisis de sangre, orina, etc.
    - **Medicamentos**: Contiene las recetas prescritas por los medicos, tambien incluye un registro de cuando y como se ha administrado el medicamiento.
    - **Facturación y Diagnósticos**: Inluye los datos que indican los codigos de los diagnosticos y procedimientos realizados.

- **icu**: Contiene **94,458** registros de estancias de pacientes en la UCI. Estos datos se estructuran de forma que hay una tabla "estancias" central que se conecta con otras tablas de eventos, estas tablas registran todos los sucesos de un paciente minuto a minuto.

Con este conocimiento solo queda conocer los campos que enlazan las diferentes tablas, los campos mas importantes son:

- `subject_id`: Identificador unico del paciente
- `hadm_id`: Identificador unico de admision hospitalaria (un paciente puede tener mas de una admision)
- `stay_id`: Identificador unico de estancia en la UCI (una admission puede tener varias estancias)

In [3]:
ID_SUBJECT = 'subject_id'
ID_ADMISION = 'hadm_id'
ID_STAY = 'stay_id'

El primer paso se tratara de analizar los diagnosticos del dataset usando la variable `icd_code`, esta variable nos indica el codigo ICD del diagnostico, este codigo usa un sistema estandarizado para codificar enfermedades, signos o sintomas, actualmente existen dos principales revisiones: **ICD-9** y **ICD-10**. Estas dos revisiones varian sobretodo en la cantidad de codigos y estructura, el objetivo es usar estos codigos para identificar en el dataset los diagnosis que contengan casos de sepsis, o algun sintoma o enfermedad que nos ayude a indicar la sepsis, se usara la variable `icd_version` para identificar la version en los datos.


### ICD-9

| Código(s) | Descripción |
|-----------|-------------|
| **995.91** | Sepsis (SIRS de origen infeccioso) |
| **995.92** | Sepsis grave |
| **785.52** | Choque séptico |
| **038.0–038.9** | Septicemia (038.11 Staph aureus, 038.42 E. coli, 038.9 no especificada) |
| **771.81** | Sepsis neonatal |

### ICD-10

| Código(s) | Descripción |
|-----------|-------------|
| **A41.9** | Sepsis, organismo no especificado (el más común en registros) |
| **A40.0, A40.1, A40.3, A40.8, A40.9** | Sepsis por diferentes grupos de Estreptococos |
| **A41.01** | Sepsis por *Staphylococcus aureus* sensible a meticilina (MSSA) |
| **A41.02** | Sepsis por *Staphylococcus aureus* resistente a meticilina (MRSA) |
| **A41.51** | Sepsis por *Escherichia coli* |
| **A41.52** | Sepsis por *Pseudomonas* |
| **A41.81** | Sepsis por *Enterococcus* |
| **A41.89** | Sepsis por otros organismos especificados (incluye sepsis viral) |
| **A02.1** | Sepsis por *Salmonella* |
| **B37.7** | Sepsis candidiásica (fúngica) |
| **R65.20** | Sepsis grave sin choque séptico |
| **R65.21** | Sepsis grave con choque séptico |
| **P36.0–P36.9** | Sepsis bacteriana neonatal |
| **O85** | Sepsis puerperal (postparto) |
| **O75.3** | Sepsis durante el trabajo de parto |
| **T81.44** | Sepsis tras procedimiento quirúrgico |

In [4]:
diagnoses = pd.read_csv('../data/hosp/diagnoses_icd.csv.gz')
print(diagnoses.shape)
print(diagnoses.dtypes)
print(diagnoses.head())

(6364488, 5)
subject_id     int64
hadm_id        int64
seq_num        int64
icd_code         str
icd_version    int64
dtype: object
   subject_id   hadm_id  seq_num icd_code  icd_version
0    10000032  22595853        1     5723            9
1    10000032  22595853        2    78959            9
2    10000032  22595853        3     5715            9
3    10000032  22595853        4    07070            9
4    10000032  22595853        5      496            9


In [5]:
ICD_CODE = 'icd_code' # Identificador del codigo de ICD 
ICD_VERSION = 'icd_version' # Identificador de la version de ICD

# ---------------------------------- ICD 9 ----------------------------------

# Definimos los codigos especificos de ICD9
SEPSIS_ICD9_EXPLICIT = {
    '99591',  # Sepsis
    '99592',  # Sepsis grave
    '78552',  # Choque séptico
    '77181',  # Sepsis neonatal
}

# Filtramos por version
icd9_df = diagnoses[diagnoses[ICD_VERSION] == 9].copy()

# Incluimos los diagnosis con codigo que empieza por 038 (Septicemia)
icd9_mask = (
    icd9_df[ICD_CODE].isin(SEPSIS_ICD9_EXPLICIT) |
    icd9_df[ICD_CODE].str.startswith('038')
)

# Evitamos registros duplicados identificandolos por admision
sepsis_icd9_hadm = icd9_df[icd9_mask][ID_ADMISION].unique()

# ---------------------------------- ICD 10 ----------------------------------

# Prefijos A40.x y A41.x, se capturan todos los subtipos
ICD10_SEPSIS_PREFIXES = ('A40', 'A41')

# Códigos adicionales fuera de A40/A41
ICD10_SEPSIS_EXTRA = {
    'A021',   # Salmonella sepsis
    'B377',   # Candida sepsis (fúngica)
    'R6520',  # Sepsis grave sin choque séptico
    'R6521',  # Sepsis grave con choque séptico
    'O85',    # Sepsis puerperal
    'O753',   # Sepsis durante el trabajo de parto
    'T8144',  # Sepsis postquirúrgica
}

# Prefijos de sepsis neonatal P36.x
ICD10_NEONATAL_PREFIX = ('P36',)

# Filtramos por version y incluimos todos los codigos
icd10_mask = (
    (diagnoses[ICD_VERSION] == 10) &
    (
        diagnoses[ICD_CODE].str.startswith(ICD10_SEPSIS_PREFIXES) |
        diagnoses[ICD_CODE].str.startswith(ICD10_NEONATAL_PREFIX) |
        diagnoses[ICD_CODE].isin(ICD10_SEPSIS_EXTRA)
    )
)

# Evitamos registros duplicados identificandolos por admision
sepsis_icd10_hadm = diagnoses[icd10_mask][ID_ADMISION].unique()

# Unimos todos los registros que contienen ICD9 y ICD10
sepsis_hadm_ids = np.union1d(sepsis_icd10_hadm, sepsis_icd9_hadm)

print(f'Admisiones con sepsis (ICD-9):  {len(sepsis_icd9_hadm):,}')
print(f'Admisiones con sepsis (ICD-10): {len(sepsis_icd10_hadm):,}')
print(f'Total admisiones:        {len(sepsis_hadm_ids):,}')

Admisiones con sepsis (ICD-9):  8,705
Admisiones con sepsis (ICD-10): 13,763
Total admisiones:        22,467


Despues de indicar los codigos y versiones ICD se puede observar que hay un total de **22,467** resultados de admisiones, teniendo en cuenta el total de admisiones se puede inferir que solo indicar los codigos ICD no es suficiente, esto se debe a que se estan capturando solo los casos donde la sepsis esta codificada de forma explicita, pero puede suceder que haya muchos casos que no esten codificados explicitamente y tengan otros indicadores que nos puede ayudar a llegar a la conclusion de que se trata de una sepsis. Para acabar de completar los datos es posible seguir dos estrategias que ya son fiables y un estandar en el sector medico:

- **Algoritmo Angus**: Muchos casos de sepsis en ICD-9 eran registrados como **Infección + Disfunción orgánica**, al usar estos indicadores juntos es posible obtener mas resultados de diagnosis de sepsis.

- **Sepsis-3**: Requiere usar el indicador **SOFA** junto a si el diagnosis entra dentro de la categoria de infeccion, al usar este calculo es posible filtrar casos de sepsis, implementarlo implica anadir mucha complejidad al analisis.

Para el primer analisis se usara solo el algoritmo angus para no anadir demasiada complejidad.

In [6]:
# ---------------------------------- Algoritmo Angus ----------------------------------

#
ANGUS_SINGLE_CODES = {'590', '597', '5990', '6816', '9966', '9985', '9993'}

# rangos de infección
ANGUS_INFECTION_RANGES = [
    (1,   41),   # Infecciones bacterianas y sistémicas
    (90,  104),  # Sífilis y espiroquetas
    (110, 118),  # Micosis
    (320, 325),  # Infecciones SNC
    (461, 486),  # Infecciones respiratorias / neumonía
    (540, 542),  # Apendicitis / abdomen
    (566, 567),  # Peritonitis
]

def is_icd9_angus(code: str) -> bool:
    """Devuelve True si el codigo entra dentro del rango de infeccion definido por el algoritmo Angus"""
    try:
        numeric = int(code[:3])
        return any(lo <= numeric <= hi for lo, hi in ANGUS_INFECTION_RANGES)
    except:
        return False


icd9_angus_df = diagnoses[diagnoses[ICD_VERSION] == 9].copy()
    
icd9_angus_mask = (
    icd9_angus_df[ICD_CODE].isin(ANGUS_SINGLE_CODES) |
    icd9_angus_df[ICD_CODE].apply(is_icd9_angus)
)

icd9_angus_df = icd9_angus_df[icd9_angus_mask][ID_ADMISION].unique()
sepsis_hadm_ids = np.union1d(sepsis_icd10_hadm, icd9_angus_df)

print(f'Admisiones con sepsis con algoritmo Angus (ICD-9): {len(icd9_angus_df):,}')
print(f"Total admisiones:        {len(sepsis_hadm_ids):,}")

Admisiones con sepsis con algoritmo Angus (ICD-9): 63,066
Total admisiones:        76,828


Al aplicar el algoritmo Angus se consigue un total de **76,828** admisiones. Ahora que tenemos los diagnosticos de las admisiones nos falta determinar el momento exacto en el que el paciente desarrolla la patologia. Es necesario observar cuando hay por ejemplo un incremento de dos puntos en el score SOFA, para conseguir este objetivo es necesario cruzar la tabla de diagnosis con las tablas `transfers` y `icustays`. Las tablas establecen una linea de tiempo que nos permite observar los eventos previos a parte de cuando ocurre un incremento en los valores criticos.

Aunque ahora hay una definicion de las dos tablas necesarias para establecer una linea de tiempo es necesario anadir otra tabla que ayuda a identificar la admision y terminar de crear la conexion entre las diferentes tablas, la tabla que se usara para este proposito es la tabla `admissions`.

In [7]:
admissions = pd.read_csv('../data/hosp/admissions.csv.gz')
transfers = pd.read_csv('../data/hosp/transfers.csv.gz')
icustays = pd.read_csv('../data/icu/icustays.csv.gz')

print('-------------------------------- Tabla admissions --------------------------------')
print(admissions.shape)
print(admissions.dtypes)

print('-------------------------------- Tabla icustays --------------------------------')
print(icustays.shape)
print(icustays.dtypes)

print('-------------------------------- Tabla transfers --------------------------------')
print(transfers.shape)
print(transfers.dtypes)

-------------------------------- Tabla admissions --------------------------------
(546028, 16)
subject_id              int64
hadm_id                 int64
admittime                 str
dischtime                 str
deathtime                 str
admission_type            str
admit_provider_id         str
admission_location        str
discharge_location        str
insurance                 str
language                  str
marital_status            str
race                      str
edregtime                 str
edouttime                 str
hospital_expire_flag    int64
dtype: object
-------------------------------- Tabla icustays --------------------------------
(94458, 8)
subject_id          int64
hadm_id             int64
stay_id             int64
first_careunit        str
last_careunit         str
intime                str
outtime               str
los               float64
dtype: object
-------------------------------- Tabla transfers --------------------------------
(2413581, 7)
s

Despues de cargar las dos tablas se puede observar que las dos contienen la variable `hadm_id` que nos permite identificar la admision, aparte de las otras variables que ayudan a identificar la estancia y el paciente (`stay_id` y `subject_id`). El siguiente paso es enlazar las diferentes tablas que ya se han cargado.

In [9]:
sepsis_df = pd.DataFrame({ID_ADMISION: list(sepsis_hadm_ids)})

cohort_icu = sepsis_df.merge(
    admissions[['subject_id', ID_ADMISION, 'admittime']],
    on=ID_ADMISION,
    how='inner'
)

final_cohort = cohort_icu.merge(
    icustays[['subject_id', ID_ADMISION, ID_STAY, 'intime', 'outtime']], 
    on=['subject_id', ID_ADMISION],
    how='inner'
)

print(final_cohort.shape)
print(final_cohort.dtypes)
print(final_cohort.head())

(29966, 6)
hadm_id       int64
subject_id    int64
admittime       str
stay_id       int64
intime          str
outtime         str
dtype: object
    hadm_id  subject_id            admittime   stay_id               intime  \
0  20002950    18596567  2134-06-21 20:57:00  32853022  2134-06-21 21:52:00   
1  20003014    15272409  2146-05-07 17:50:00  34079238  2146-05-07 23:01:50   
2  20003427    13927771  2184-05-14 20:15:00  35250851  2184-05-23 20:20:50   
3  20003427    13927771  2184-05-14 20:15:00  35475938  2184-05-14 21:45:00   
4  20003491    11540283  2197-12-18 04:50:00  33494445  2197-12-18 06:10:00   

               outtime  
0  2134-06-22 16:25:45  
1  2146-05-09 03:32:21  
2  2184-05-24 02:10:52  
3  2184-05-23 20:20:28  
4  2197-12-20 19:02:58  


Una vez unidas las tablas de estancias y transferencias se puede observar que obtenemos un total de **29,966 registros**, esta tabla contiene la fecha y hora de entrada y salida del paciente por cada admision, incluyendo tambien la estancia. El siguiente paso es extraer las variables temporales para construir la **memoria** del modelo. El objetivo es priorizar la extraccion de variables que permiten identificar mejor la patologia, por ejemplo el score **SOFA**, este es el estandar clinico de referencia que se puede usar. Existen dos tablas que se pueden usar para este proposito:

- **chartevents**: Contiene valores de los signos vitales como la frecuencia cardiaca, presion arterial media, saturacion de oxigeno y temperatura.

- **labevents**: Incluye valores como el lactato, creatinina, bilirrubina, recuento de plaquetas y relacion PaO2/FiO2.